In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# train_v8_convnextv2_fcmae_wandb_resume_colab

Автономный Colab notebook для продолжения `ConvNeXt V2 + LSS` обучения из `wandb` artifacts.

Что делает:
- работает с kaggle-safe dataset paths без `:`
- скачивает `best.pt`, `ema_best.pt`, `last.pt` из `wandb`
- резюмится из `last.pt`
- продолжает обучение ещё на `10` эпох
- логирует метрики в `wandb`
- загружает bundle checkpoint artifact **каждую эпоху**


In [ ]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, gc, uuid, subprocess, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from PIL import Image, ImageFile, ImageFilter
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Kaggle-safe dataset directories in Colab
DATA_TRAIN = Path('/content/autonomy_yandex_dataset_train_kaggle_safe')
DATA_VAL   = Path('/content/autonomy_yandex_dataset_val_kaggle_safe')
DATA_TEST  = Path('/content/autonomy_yandex_dataset_test_kaggle_safe')

# Local workspace for downloaded/uploaded artifacts
ARTIFACTS_DIR = Path('/content/model_artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Weights & Biases settings
WANDB_TOKEN_PATH = Path('/content/wandb_tok') if Path('/content/wandb_tok').exists() else Path('wandb_tok')
WANDB_ENTITY = 'letit6e-hse-university'
WANDB_PROJECT = 'convnext-bev'
WANDB_RESUME_ARTIFACT_LAST = 'v8-convnext-last:latest'
WANDB_RESUME_ARTIFACT_BEST = 'v8-convnext-best:latest'
WANDB_RESUME_ARTIFACT_EMA_BEST = 'v8-convnext-ema-best:latest'

cfg = {
    'run_dir': '/content/runs/v8_convnextv2_fcmae_wandb_resume_colab',
    'img_hw': (384, 704),
    'batch_size': 2,
    'val_batch_size': 1,
    'grad_accum': 8,
    'epochs': 10,
    'warmup_epochs': 0,
    'freeze_backbone_epochs': 0,
    'unfreeze_last_stages': 2,
    'lr_backbone': 2.0e-5,
    'lr_image_neck': 1.4e-4,
    'lr_lss_bev': 2.0e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'fpn_dim': 128,
    'context_dim': 80,
    'depth_bins': 24,
    'depth_min': 1.0,
    'depth_max': 80.0,
    'world_z_min': -2.0,
    'world_z_max': 4.5,
    'bev_base_channels': 96,
    'bev_gn_groups': 8,
    'convnext_model_name': 'convnextv2_base.fcmae_ft_in22k_in1k_384',
    'convnext_fallback_model_name': 'convnextv2_base.fcmae_ft_in22k_in1k',
    'use_ddp': False,
    'use_amp': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def get_world_size():
    return int(os.environ.get('WORLD_SIZE', '1'))


def get_rank():
    return int(os.environ.get('RANK', '0'))


def get_local_rank():
    return int(os.environ.get('LOCAL_RANK', '0'))


def is_dist_enabled():
    return cfg['use_ddp'] and get_world_size() > 1


def is_main_process():
    return get_rank() == 0


def setup_distributed():
    if not is_dist_enabled():
        return
    if dist.is_available() and not dist.is_initialized():
        dist.init_process_group(backend='nccl', init_method='env://')
        torch.cuda.set_device(get_local_rank())


def barrier():
    if dist.is_available() and dist.is_initialized():
        dist.barrier()


def cleanup_distributed():
    if dist.is_available() and dist.is_initialized():
        dist.destroy_process_group()


def ensure_package(pkg: str):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])


setup_distributed()

random.seed(cfg['seed'] + get_rank())
np.random.seed(cfg['seed'] + get_rank())
torch.manual_seed(cfg['seed'] + get_rank())

if torch.cuda.is_available():
    device = torch.device(f'cuda:{get_local_rank()}') if is_dist_enabled() else torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if is_main_process():
    print('device:', device)
    if torch.cuda.is_available():
        print('cuda devices visible:', torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            try:
                print(i, torch.cuda.get_device_name(i))
            except Exception as e:
                print(i, f'<cuda init error: {e}>')
    print('world_size:', get_world_size(), '| rank:', get_rank(), '| local_rank:', get_local_rank())
    print('img_hw:', cfg['img_hw'])
    print('train batch_size(per gpu):', cfg['batch_size'], '| train workers:', cfg['num_workers'])
    print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])
    print('wandb token path:', WANDB_TOKEN_PATH, '| exists:', WANDB_TOKEN_PATH.exists())
    print('dataset exists:', DATA_TRAIN.exists(), DATA_VAL.exists(), DATA_TEST.exists())

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump(json.loads(json.dumps(cfg, default=str)), f, indent=2)


In [ ]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def kaggle_safe_name(name: str) -> str:
    return name.replace(':', '_')


def resolve_info_path(base_dir: Path, p) -> Path:
    p = Path(str(p))
    candidates = []

    candidates.append(p)
    candidates.append(base_dir / p)
    candidates.append(base_dir.parent / p)

    parts = list(p.parts)
    for anchor in [
        'autonomy_yandex_dataset_train',
        'autonomy_yandex_dataset_val',
        'autonomy_yandex_dataset_test',
        'autonomy_yandex_dataset_train_kaggle_safe',
        'autonomy_yandex_dataset_val_kaggle_safe',
        'autonomy_yandex_dataset_test_kaggle_safe',
    ]:
        if anchor in parts:
            i = parts.index(anchor)
            rel = Path(*parts[i + 1:])
            candidates.append(base_dir / rel)
            candidates.append(base_dir.parent / rel)
            safe_rel = Path(*[kaggle_safe_name(x) for x in rel.parts])
            candidates.append(base_dir / safe_rel)
            candidates.append(base_dir.parent / safe_rel)

    safe_p = Path(*[kaggle_safe_name(x) for x in parts])
    candidates.append(base_dir / safe_p)
    candidates.append(base_dir.parent / safe_p)

    seen = set()
    for cand in candidates:
        cand = Path(cand)
        key = str(cand)
        if key in seen:
            continue
        seen.add(key)
        if cand.exists():
            return cand

    raise FileNotFoundError(f'Could not resolve path: {p} from base_dir={base_dir}')


def load_info_with_root(data_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(data_dir / 'info.csv', index_col=0).reset_index(drop=True).copy()
    df['__data_root'] = str(data_dir)
    df['__source_split'] = split_name
    return df


def remap_kaggle_paths(df: pd.DataFrame, train_dir: Path, val_dir: Path, test_dir: Path) -> pd.DataFrame:
    df = df.copy()
    path_cols = [*CAMERA_NAMES, *INTRINSICS_NAMES, *CAR2CAM_NAMES, GT_NAME]

    def pick_root(split: str) -> Path:
        if split == 'train':
            return train_dir
        if split == 'val':
            return val_dir
        if split == 'test':
            return test_dir
        return train_dir

    def rewrite_path(p, split: str):
        if pd.isna(p):
            return p
        p = str(p)
        pp = Path(p)
        if pp.exists():
            return str(pp)

        root = pick_root(str(split))
        parts = list(Path(p).parts)

        for anchor in [
            'autonomy_yandex_dataset_train',
            'autonomy_yandex_dataset_val',
            'autonomy_yandex_dataset_test',
            'autonomy_yandex_dataset_train_kaggle_safe',
            'autonomy_yandex_dataset_val_kaggle_safe',
            'autonomy_yandex_dataset_test_kaggle_safe',
        ]:
            if anchor in parts:
                i = parts.index(anchor)
                rel = Path(*parts[i + 1:])
                cand = root / rel
                if cand.exists():
                    return str(cand)
                safe_rel = Path(*[kaggle_safe_name(x) for x in rel.parts])
                cand = root / safe_rel
                if cand.exists():
                    return str(cand)

        safe_parts = [kaggle_safe_name(x) for x in parts]
        safe_p = Path(*safe_parts)
        cand = root / safe_p
        if cand.exists():
            return str(cand)

        cand = root / kaggle_safe_name(Path(p).name)
        if cand.exists():
            return str(cand)

        return p

    if '__source_split' in df.columns:
        df['__data_root'] = df['__source_split'].map({
            'train': str(train_dir),
            'val': str(val_dir),
            'test': str(test_dir),
        }).fillna(str(train_dir))
    else:
        df['__data_root'] = str(train_dir)

    for col in path_cols:
        if col in df.columns:
            df[col] = [
                rewrite_path(p, split)
                for p, split in zip(df[col], df.get('__source_split', pd.Series(['train'] * len(df))))
            ]

    return df


def resolve_row_path(row: pd.Series, key: str) -> Path:
    return resolve_info_path(Path(row['__data_root']), row[key])


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [ ]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()
train_info = remap_kaggle_paths(train_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(val_info, DATA_TRAIN, DATA_VAL, DATA_TEST)

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))

if is_main_process() and len(val_info):
    row = val_info.iloc[0]
    for key in [CAMERA_NAMES[0], INTRINSICS_NAMES[0], CAR2CAM_NAMES[0], GT_NAME]:
        path = resolve_info_path(Path(row['__data_root']), row[key])
        print(key, path, path.exists())


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
def gn_groups(channels: int, requested: int = 8) -> int:
    g = min(requested, channels)
    while channels % g != 0 and g > 1:
        g -= 1
    return max(g, 1)


class ConvGNAct(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, groups=8, act=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_c, out_c, k, stride=s, padding=p, bias=False),
            nn.GroupNorm(gn_groups(out_c, groups), out_c),
        ]
        if act:
            layers.append(nn.SiLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class ResidualBlock2d(nn.Module):
    def __init__(self, in_c, out_c, stride=1, groups=8):
        super().__init__()
        self.conv1 = ConvGNAct(in_c, out_c, k=3, s=stride, p=1, groups=groups, act=True)
        self.conv2 = ConvGNAct(out_c, out_c, k=3, s=1, p=1, groups=groups, act=False)
        if stride != 1 or in_c != out_c:
            self.skip = ConvGNAct(in_c, out_c, k=1, s=stride, p=0, groups=groups, act=False)
        else:
            self.skip = nn.Identity()
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)) + self.skip(x))


class ASPP2d(nn.Module):
    def __init__(self, in_c, out_c, rates=(1, 3, 6), groups=8):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=r, dilation=r, bias=False),
                nn.GroupNorm(gn_groups(out_c, groups), out_c),
                nn.SiLU(inplace=True),
            )
            for r in rates
        ])
        self.proj = ConvGNAct(out_c * len(rates), out_c, k=1, s=1, p=0, groups=groups, act=True)

    def forward(self, x):
        xs = [b(x) for b in self.branches]
        return self.proj(torch.cat(xs, dim=1))


class ConvBNAct(nn.Module):
    def __init__(self, in_c, out_c, k, stride=1, padding=None, groups=1,
                 bias=False, eps=1e-3, momentum=0.01, act=True):
        super().__init__()
        if padding is None:
            padding = k // 2
        self.conv = nn.Conv2d(in_c, out_c, k, stride=stride, padding=padding, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_c, eps=eps, momentum=momentum)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ChannelAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Conv2d(channels, channels, 1, 1, 0, bias=True)
        self.act = nn.Hardsigmoid(inplace=True)

    def forward(self, x):
        with torch.cuda.amp.autocast(enabled=False):
            a = self.pool(x.float())
            a = self.fc(a)
            a = self.act(a)
        return x * a.to(dtype=x.dtype)


class SPPBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(5, 9, 13), eps=1e-3, momentum=0.01):
        super().__init__()
        mid_channels = in_channels // 2
        self.conv1 = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.poolings = nn.ModuleList([
            nn.MaxPool2d(kernel_size=ks, stride=1, padding=ks // 2) for ks in kernel_sizes
        ])
        self.conv2 = ConvBNAct(mid_channels * (len(kernel_sizes) + 1), out_channels, 1, eps=eps, momentum=momentum)

    def forward(self, x):
        x = self.conv1(x)
        with torch.cuda.amp.autocast(enabled=False):
            x32 = x.float()
            x = torch.cat([x32] + [pool(x32) for pool in self.poolings], dim=1)
        x = x.to(dtype=self.conv2.conv.weight.dtype)
        return self.conv2(x)


class CSPNeXtBlock(nn.Module):
    def __init__(self, in_channels, out_channels, expansion=0.5, add_identity=True, kernel_size=5,
                 eps=1e-3, momentum=0.01):
        super().__init__()
        hidden_channels = int(out_channels * expansion)
        self.conv1 = ConvBNAct(in_channels, hidden_channels, 3, eps=eps, momentum=momentum)
        self.conv2 = nn.Sequential(
            ConvBNAct(hidden_channels, hidden_channels, kernel_size, groups=hidden_channels, eps=eps, momentum=momentum),
            ConvBNAct(hidden_channels, out_channels, 1, eps=eps, momentum=momentum),
        )
        self.add_identity = add_identity and in_channels == out_channels

    def forward(self, x):
        y = self.conv1(x)
        y = self.conv2(y)
        return y + x if self.add_identity else y


class CSPLayer(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio=0.5, num_blocks=1,
                 add_identity=True, channel_attention=True, eps=1e-3, momentum=0.01):
        super().__init__()
        mid_channels = int(out_channels * expand_ratio)
        self.main_conv = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.short_conv = ConvBNAct(in_channels, mid_channels, 1, eps=eps, momentum=momentum)
        self.final_conv = ConvBNAct(2 * mid_channels, out_channels, 1, eps=eps, momentum=momentum)
        self.blocks = nn.Sequential(*[
            CSPNeXtBlock(mid_channels, mid_channels, expansion=1.0, add_identity=add_identity, eps=eps, momentum=momentum)
            for _ in range(num_blocks)
        ])
        self.attention = ChannelAttention(2 * mid_channels) if channel_attention else nn.Identity()

    def forward(self, x):
        x_short = self.short_conv(x)
        x_main = self.main_conv(x)
        x_main = self.blocks(x_main)
        x = torch.cat((x_main, x_short), dim=1)
        x = self.attention(x)
        return self.final_conv(x)


class CSPNeXtBackboneFromRTMDet(nn.Module):
    arch_settings = {
        'P5': [
            [64, 128, 3, True, False],
            [128, 256, 6, True, False],
            [256, 512, 6, True, False],
            [512, 1024, 3, False, True],
        ]
    }

    def __init__(self, arch='P5', deepen_factor=1.0, widen_factor=1.0,
                 expand_ratio=0.5, channel_attention=True,
                 out_indices=(2, 3, 4), eps=1e-3, momentum=0.01):
        super().__init__()
        arch_setting = self.arch_settings[arch]
        self.out_indices = out_indices
        c0 = int(arch_setting[0][0] * widen_factor)
        self.stem = nn.Sequential(
            ConvBNAct(3, c0 // 2, 3, stride=2, eps=eps, momentum=momentum),
            ConvBNAct(c0 // 2, c0 // 2, 3, stride=1, eps=eps, momentum=momentum),
            ConvBNAct(c0 // 2, c0, 3, stride=1, eps=eps, momentum=momentum),
        )
        self.layers = ['stem']
        for i, (in_c, out_c, num_blocks, add_identity, use_spp) in enumerate(arch_setting):
            in_c = int(in_c * widen_factor)
            out_c = int(out_c * widen_factor)
            num_blocks = max(round(num_blocks * deepen_factor), 1)
            stage = [ConvBNAct(in_c, out_c, 3, stride=2, eps=eps, momentum=momentum)]
            if use_spp:
                stage.append(SPPBottleneck(out_c, out_c, eps=eps, momentum=momentum))
            stage.append(CSPLayer(
                out_c, out_c,
                expand_ratio=expand_ratio,
                num_blocks=num_blocks,
                add_identity=add_identity,
                channel_attention=channel_attention,
                eps=eps,
                momentum=momentum,
            ))
            self.add_module(f'stage{i + 1}', nn.Sequential(*stage))
            self.layers.append(f'stage{i + 1}')

    def forward(self, x):
        outs = []
        for i, layer_name in enumerate(self.layers):
            layer = getattr(self, layer_name)
            x = layer(x)
            if i in self.out_indices:
                outs.append(x)
        return tuple(outs)



def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if 'state_dict' in ckpt:
            return ckpt['state_dict']
        if 'model' in ckpt and isinstance(ckpt['model'], dict):
            return ckpt['model']
    return ckpt



def load_rtmdet_pretrained_backbone(backbone: nn.Module, ckpt_path: Path):
    if not Path(ckpt_path).exists():
        raise FileNotFoundError(ckpt_path)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    state_dict = extract_state_dict(ckpt)

    def remap_key(k: str):
        if not k.startswith('backbone.'):
            return None
        k = k[len('backbone.'):]
        k = k.replace('.conv2.depthwise_conv.', '.conv2.0.')
        k = k.replace('.conv2.pointwise_conv.', '.conv2.1.')
        return k

    filtered = {}
    remapped = 0
    for k, v in state_dict.items():
        new_k = remap_key(k)
        if new_k is None:
            continue
        if new_k != k[len('backbone.'):]:
            remapped += 1
        filtered[new_k] = v

    missing, unexpected = backbone.load_state_dict(filtered, strict=False)
    loaded_keys = set(filtered.keys()) - set(unexpected)
    summary = {
        'checkpoint': str(ckpt_path),
        'raw_keys': len(state_dict),
        'backbone_candidate_keys': len(filtered),
        'remapped_keys': remapped,
        'loaded_keys': len(loaded_keys),
        'missing_keys': len(missing),
        'unexpected_keys': len(unexpected),
    }
    print(json.dumps(summary, indent=2))
    if len(missing):
        print('sample missing:', missing[:20])
    if len(unexpected):
        print('sample unexpected:', unexpected[:20])
    return summary


class _RTMDetMultiScaleBackbone(nn.Module):
    def __init__(self,
                 pretrained_backbone_path: str,
                 arch='P5',
                 deepen_factor=1.0,
                 widen_factor=1.0,
                 expand_ratio=0.5,
                 channel_attention=True,
                 out_indices=(2, 3, 4),
                 fpn_dim: int = 128,
                 groups: int = 8,
                 eps=1e-3,
                 momentum=0.01):
        super().__init__()
        self.fpn_dim = fpn_dim
        self.backbone = CSPNeXtBackboneFromRTMDet(
            arch=arch,
            deepen_factor=deepen_factor,
            widen_factor=widen_factor,
            expand_ratio=expand_ratio,
            channel_attention=channel_attention,
            out_indices=out_indices,
            eps=eps,
            momentum=momentum,
        )
        self.backbone_load_summary = load_rtmdet_pretrained_backbone(self.backbone, Path(pretrained_backbone_path))
        self.laterals = nn.ModuleList([
            nn.Conv2d(256, fpn_dim, 1),
            nn.Conv2d(512, fpn_dim, 1),
            nn.Conv2d(1024, fpn_dim, 1),
        ])
        self.smooth16 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.smooth8 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.out_proj = nn.Sequential(
            ConvGNAct(fpn_dim * 3, fpn_dim, k=1, s=1, p=0, groups=groups, act=True),
            ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True),
        )

    def freeze_all_stages(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_stages(self, n_last_stages=2):
        self.freeze_all_stages()
        stage_names = ['stage1', 'stage2', 'stage3', 'stage4']
        for name in stage_names[-n_last_stages:]:
            stage = getattr(self.backbone, name)
            for p in stage.parameters():
                p.requires_grad = True

    def forward(self, x):
        feat_s8, feat_s16, feat_s32 = self.backbone(x)
        lat8 = self.laterals[0](feat_s8)
        lat16 = self.laterals[1](feat_s16)
        p32 = self.laterals[2](feat_s32)
        p16 = self.smooth16(lat16 + F.interpolate(p32, size=lat16.shape[-2:], mode='bilinear', align_corners=False))
        p8 = self.smooth8(lat8 + F.interpolate(p16, size=lat8.shape[-2:], mode='bilinear', align_corners=False))
        p16_up = F.interpolate(p16, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        p32_up = F.interpolate(p32, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        fused = self.out_proj(torch.cat([p8, p16_up, p32_up], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'p8': p8,
            'p16': p16,
            'p32': p32,
            'fused': fused,
        }


class LSSViewTransform2D(nn.Module):
    def __init__(self,
                 in_c: int,
                 context_c: int,
                 depth_bins: int,
                 depth_min: float,
                 depth_max: float,
                 bev_h: int,
                 bev_w: int,
                 bev_res: float,
                 x_range,
                 y_range,
                 z_min: float,
                 z_max: float,
                 groups: int = 8):
        super().__init__()
        self.context_c = context_c
        self.depth_bins = depth_bins
        self.depth_min = float(depth_min)
        self.depth_max = float(depth_max)
        self.bev_h = bev_h
        self.bev_w = bev_w
        self.bev_res = float(bev_res)
        self.x_range = x_range
        self.y_range = y_range
        self.z_min = float(z_min)
        self.z_max = float(z_max)

        self.depth_head = nn.Sequential(
            ConvGNAct(in_c, in_c, k=3, s=1, p=1, groups=groups, act=True),
            nn.Conv2d(in_c, depth_bins, 1),
        )
        self.context_head = nn.Sequential(
            ConvGNAct(in_c, in_c, k=3, s=1, p=1, groups=groups, act=True),
            nn.Conv2d(in_c, context_c, 1),
        )

    def _build_frustum(self, Hf: int, Wf: int, Hi: int, Wi: int, device, dtype):
        depths = torch.linspace(self.depth_min, self.depth_max, self.depth_bins, device=device, dtype=dtype)
        xs = (torch.arange(Wf, device=device, dtype=dtype) + 0.5) * (Wi / Wf)
        ys = (torch.arange(Hf, device=device, dtype=dtype) + 0.5) * (Hi / Hf)
        d, y, x = torch.meshgrid(depths, ys, xs, indexing='ij')
        return x, y, d

    def forward(self, feat_2d: torch.Tensor, intrinsics: torch.Tensor, car2cams: torch.Tensor, image_hw):
        B, N, C, Hf, Wf = feat_2d.shape
        Hi, Wi = image_hw

        feat_bn = feat_2d.reshape(B * N, C, Hf, Wf)
        depth_logits = self.depth_head(feat_bn).float()
        context = self.context_head(feat_bn).float()

        depth_prob = torch.softmax(depth_logits, dim=1)
        depth_prob = depth_prob.reshape(B, N, self.depth_bins, Hf, Wf)
        context = context.reshape(B, N, self.context_c, Hf, Wf)

        x_img, y_img, depth_vals = self._build_frustum(Hf, Wf, Hi, Wi, feat_2d.device, torch.float32)
        x_img = x_img.view(1, 1, self.depth_bins, Hf, Wf)
        y_img = y_img.view(1, 1, self.depth_bins, Hf, Wf)
        depth_vals = depth_vals.view(1, 1, self.depth_bins, Hf, Wf)

        intrinsics = intrinsics.float()
        car2cams = car2cams.float()
        cam2cars = torch.inverse(car2cams.reshape(B * N, 4, 4)).reshape(B, N, 4, 4)

        fx = intrinsics[..., 0, 0].view(B, N, 1, 1, 1)
        fy = intrinsics[..., 1, 1].view(B, N, 1, 1, 1)
        cx = intrinsics[..., 0, 2].view(B, N, 1, 1, 1)
        cy = intrinsics[..., 1, 2].view(B, N, 1, 1, 1)

        X = (x_img - cx) / fx * depth_vals
        Y = (y_img - cy) / fy * depth_vals
        Z = depth_vals.expand(B, N, -1, -1, -1)
        ones = torch.ones_like(Z)
        pts_cam = torch.stack([X, Y, Z, ones], dim=-1)
        pts_car = torch.einsum('bnij,bndhwj->bndhwi', cam2cars, pts_cam)

        world_x = pts_car[..., 0]
        world_y = pts_car[..., 1]
        world_z = pts_car[..., 2]

        x_idx = torch.floor((world_x - self.x_range[0]) / self.bev_res).long()
        y_idx = torch.floor((world_y - self.y_range[0]) / self.bev_res).long()
        valid = (
            (x_idx >= 0) & (x_idx < self.bev_h) &
            (y_idx >= 0) & (y_idx < self.bev_w) &
            (world_z >= self.z_min) & (world_z <= self.z_max)
        )
        linear_idx = x_idx * self.bev_w + y_idx

        feat_vol = context.unsqueeze(3) * depth_prob.unsqueeze(2)
        bev = feat_2d.new_zeros(B, self.context_c, self.bev_h * self.bev_w, dtype=torch.float32)
        counts = feat_2d.new_zeros(B, 1, self.bev_h * self.bev_w, dtype=torch.float32)

        for b in range(B):
            idx_b = linear_idx[b].reshape(-1)
            valid_b = valid[b].reshape(-1)
            if not valid_b.any():
                continue
            feat_b = feat_vol[b].permute(1, 0, 2, 3, 4).reshape(self.context_c, -1)
            idx_valid = idx_b[valid_b]
            feat_valid = feat_b[:, valid_b]
            bev[b].scatter_add_(1, idx_valid.unsqueeze(0).expand(self.context_c, -1), feat_valid)
            counts[b].scatter_add_(1, idx_valid.unsqueeze(0), torch.ones(1, idx_valid.numel(), device=feat_2d.device, dtype=torch.float32))

        bev = bev / counts.clamp(min=1.0)
        bev = bev.reshape(B, self.context_c, self.bev_h, self.bev_w)
        bev = torch.nan_to_num(bev, nan=0.0, posinf=0.0, neginf=0.0)

        debug = {
            'depth_logits': depth_logits.reshape(B, N, self.depth_bins, Hf, Wf),
            'depth_prob': depth_prob,
            'context': context,
            'bev': bev,
            'valid_ratio': valid.float().mean().item(),
        }
        return bev, debug


class StrongBEVEncoderDecoder(nn.Module):
    def __init__(self, in_c: int, base_c: int = 96, groups: int = 8):
        super().__init__()
        self.stem = nn.Sequential(
            ConvGNAct(in_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.down1 = nn.Sequential(
            ResidualBlock2d(base_c, base_c * 2, stride=2, groups=groups),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.down2 = nn.Sequential(
            ResidualBlock2d(base_c * 2, base_c * 4, stride=2, groups=groups),
            ResidualBlock2d(base_c * 4, base_c * 4, stride=1, groups=groups),
        )
        self.aspp = ASPP2d(base_c * 4, base_c * 4, rates=(1, 3, 6), groups=groups)
        self.up1 = nn.Sequential(
            ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c * 2, base_c * 2, stride=1, groups=groups),
        )
        self.up0 = nn.Sequential(
            ConvGNAct(base_c * 2 + base_c, base_c, k=3, s=1, p=1, groups=groups, act=True),
            ResidualBlock2d(base_c, base_c, stride=1, groups=groups),
        )
        self.head = nn.Conv2d(base_c, 1, 1)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        return self.head(u0)


class MultiCamBEVv7RTMDetCSPNeXtLSSClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 pretrained_backbone_path: str = './rtmdet_l_8xb32-300e_coco_20220719_112030-5a0be7c4.pth',
                 csp_arch: str = 'P5',
                 csp_deepen_factor: float = 1.0,
                 csp_widen_factor: float = 1.0,
                 csp_expand_ratio: float = 0.5,
                 csp_channel_attention: bool = True,
                 csp_out_indices=(2, 3, 4),
                 fpn_dim: int = 128,
                 context_dim: int = 80,
                 depth_bins: int = 24,
                 depth_min: float = 1.0,
                 depth_max: float = 80.0,
                 world_z_min: float = -2.0,
                 world_z_max: float = 4.5,
                 bev_base_channels: int = 96,
                 bev_gn_groups: int = 8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _RTMDetMultiScaleBackbone(
            pretrained_backbone_path=pretrained_backbone_path,
            arch=csp_arch,
            deepen_factor=csp_deepen_factor,
            widen_factor=csp_widen_factor,
            expand_ratio=csp_expand_ratio,
            channel_attention=csp_channel_attention,
            out_indices=csp_out_indices,
            fpn_dim=fpn_dim,
            groups=bev_gn_groups,
        )
        if freeze_backbone:
            self.backbone.freeze_all_stages()

        self.view_transform = LSSViewTransform2D(
            in_c=fpn_dim,
            context_c=context_dim,
            depth_bins=depth_bins,
            depth_min=depth_min,
            depth_max=depth_max,
            bev_h=BEV_H,
            bev_w=BEV_W,
            bev_res=BEV_RES,
            x_range=X_RANGE,
            y_range=Y_RANGE,
            z_min=world_z_min,
            z_max=world_z_max,
            groups=bev_gn_groups,
        )

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        self.bev_decoder = StrongBEVEncoderDecoder(
            in_c=context_dim + rover_cond_dim,
            base_c=bev_base_channels,
            groups=bev_gn_groups,
        )

    def forward_debug(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        assert N == self.n_cameras
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        feat_s8 = back['feat_s8'].reshape(B, N, back['feat_s8'].shape[1], back['feat_s8'].shape[2], back['feat_s8'].shape[3])
        feat_s16 = back['feat_s16'].reshape(B, N, back['feat_s16'].shape[1], back['feat_s16'].shape[2], back['feat_s16'].shape[3])
        feat_s32 = back['feat_s32'].reshape(B, N, back['feat_s32'].shape[1], back['feat_s32'].shape[2], back['feat_s32'].shape[3])
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev, vt_debug = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'image_fused': fused,
            'depth_logits': vt_debug['depth_logits'],
            'depth_prob': vt_debug['depth_prob'],
            'bev_raw': vt_debug['bev'],
            'valid_ratio': vt_debug['valid_ratio'],
            'logits': logits,
        }

    def forward(self, images, intrinsics, car2cams, rover_ids):
        dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
        logits = torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)
        return logits



def load_resume_state(core_model, ema_model, optimizer, scheduler, scaler, run_dir: Path):
    resume_ckpt = Path(cfg.get('resume_ckpt', ''))
    if not cfg.get('resume_training', False) or not resume_ckpt.exists():
        return {
            'enabled': False,
            'start_epoch': 0,
            'best_iou': -1.0,
            'best_ema_iou': -1.0,
            'log': [],
            'elapsed_minutes': 0.0,
        }

    ckpt = torch.load(resume_ckpt, map_location='cpu')
    core_model.load_state_dict(ckpt['model'], strict=False)
    if 'ema' in ckpt:
        ema_model.load_state_dict(ckpt['ema'], strict=False)
    if 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    if 'scheduler' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler'])
    if 'scaler' in ckpt and ckpt['scaler'] is not None:
        scaler.load_state_dict(ckpt['scaler'])

    start_epoch = int(ckpt.get('epoch', -1)) + 1
    log_path = run_dir / 'log.csv'
    log_rows = []
    elapsed_minutes = 0.0
    if log_path.exists():
        log_rows = pd.read_csv(log_path).to_dict('records')
        if len(log_rows):
            elapsed_minutes = float(log_rows[-1].get('minutes', 0.0) or 0.0)

    best_iou = float(ckpt.get('best_iou', -1.0))
    best_ema_iou = float(ckpt.get('best_ema_iou', -1.0))
    best_path = run_dir / 'best.pt'
    ema_best_path = run_dir / 'ema_best.pt'
    if best_path.exists():
        try:
            best_iou = max(best_iou, float(torch.load(best_path, map_location='cpu').get('best_iou', -1.0)))
        except Exception:
            pass
    if ema_best_path.exists():
        try:
            best_ema_iou = max(best_ema_iou, float(torch.load(ema_best_path, map_location='cpu').get('best_ema_iou', -1.0)))
        except Exception:
            pass

    print('resumed from', resume_ckpt)
    print('  start_epoch:', start_epoch)
    print('  best_iou so far:', best_iou)
    print('  best_ema_iou so far:', best_ema_iou)
    print('  prior log rows:', len(log_rows))
    return {
        'enabled': True,
        'start_epoch': start_epoch,
        'best_iou': best_iou,
        'best_ema_iou': best_ema_iou,
        'log': log_rows,
        'elapsed_minutes': elapsed_minutes,
    }


In [ ]:
def _lovasz_grad(gt_sorted: torch.Tensor) -> torch.Tensor:
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    if labels.numel() == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CompoundLossV2(nn.Module):
    def __init__(self, pos_weight: float = 5.0,
                 weight_bce: float = 0.5,
                 weight_dice: float = 0.3,
                 weight_lovasz: float = 0.2,
                 ignore_value: int = 255):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz
        self.ignore_value = ignore_value

    def forward(self, logits: torch.Tensor, gt: torch.Tensor):
        logits = logits.float()
        gt = gt.long()
        valid = (gt != self.ignore_value)
        gt_f = (gt == 1).float()

        bce_per = F.binary_cross_entropy_with_logits(logits, gt_f, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)

        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_f * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()

        lov_logits = logits[valid]
        lov_gt = gt_f[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0

        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        parts = {'bce': float(bce.item()), 'dice': float(dice.item()), 'lovasz': float(lov.item())}
        return total, parts


def unwrap_model(model):
    return model.module if hasattr(model, 'module') else model


@torch.no_grad()
def iou_binary_batch(logits: torch.Tensor, gt: torch.Tensor, threshold: float = 0.5, ignore_value: int = 255):
    probs = torch.sigmoid(logits)
    pred = probs > threshold
    valid = gt != ignore_value
    gt_b = (gt == 1) & valid
    pred = pred & valid
    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter, union


@torch.no_grad()
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.inference_mode()
def evaluate_iou(model, loader, threshold=0.5, desc='val@0.5'):
    model.eval()
    inter = 0
    union = 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        i, u = iou_binary_batch(logits, gt, threshold=threshold)
        inter += i
        union += u
        pbar.set_postfix(iou=f'{inter / max(union, 1):.4f}')
    return inter / max(union, 1)


In [ ]:
ensure_package('timm')
ensure_package('safetensors')
ensure_package('wandb')

import timm
import wandb
from safetensors.torch import load_file as safe_load_file

if not WANDB_TOKEN_PATH.exists():
    raise FileNotFoundError(f'Missing wandb token file: {WANDB_TOKEN_PATH}')

token = WANDB_TOKEN_PATH.read_text().strip().strip('"').strip("'")
os.environ['WANDB_API_KEY'] = token
wandb.login(relogin=True)

api = wandb.Api()

WAND_B_DOWNLOAD_DIR = ARTIFACTS_DIR / 'wandb_downloads'
WAND_B_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

def download_artifact_to_dir(artifact_ref: str, target_dir: Path):
    artifact = api.artifact(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{artifact_ref}", type='model')
    local_dir = Path(artifact.download(root=str(target_dir / artifact_ref.replace(':', '__'))))
    return artifact, local_dir

last_artifact, last_dir = download_artifact_to_dir(WANDB_RESUME_ARTIFACT_LAST, WAND_B_DOWNLOAD_DIR)
best_artifact, best_dir = download_artifact_to_dir(WANDB_RESUME_ARTIFACT_BEST, WAND_B_DOWNLOAD_DIR)
ema_artifact, ema_dir = download_artifact_to_dir(WANDB_RESUME_ARTIFACT_EMA_BEST, WAND_B_DOWNLOAD_DIR)

CONVNEXT_RESUME_LAST = next(last_dir.rglob('last.pt'))
CONVNEXT_RESUME_BEST = next(best_dir.rglob('best.pt'))
CONVNEXT_RESUME_EMA_BEST = next(ema_dir.rglob('ema_best.pt'))

print('downloaded last:', CONVNEXT_RESUME_LAST)
print('downloaded best:', CONVNEXT_RESUME_BEST)
print('downloaded ema_best:', CONVNEXT_RESUME_EMA_BEST)

# Keep local copies in ARTIFACTS_DIR for convenience
local_last = ARTIFACTS_DIR / 'v8_last.pt'
local_best = ARTIFACTS_DIR / 'v8_best.pt'
local_ema = ARTIFACTS_DIR / 'v8_ema_best.pt'
for src, dst in [
    (CONVNEXT_RESUME_LAST, local_last),
    (CONVNEXT_RESUME_BEST, local_best),
    (CONVNEXT_RESUME_EMA_BEST, local_ema),
]:
    dst.write_bytes(src.read_bytes())

resume_probe = torch.load(local_last, map_location='cpu')
print('resume epoch:', resume_probe.get('epoch'))
print('resume model_type:', resume_probe.get('model_type'))
print('resume best_iou:', resume_probe.get('best_iou'))
print('resume best_ema_iou:', resume_probe.get('best_ema_iou'))


In [ ]:
def _strip_prefix_if_present(state_dict, prefix: str):
    out = {}
    hit = False
    for k, v in state_dict.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
            hit = True
    return out if hit else None


def flexible_load_into_module(module: nn.Module, state_dict: dict):
    current = module.state_dict()
    variants = [('raw', state_dict)]
    for prefix in ['model.', 'module.', 'backbone.', 'model.backbone.', 'module.backbone.']:
        candidate = _strip_prefix_if_present(state_dict, prefix)
        if candidate is not None:
            variants.append((f'strip:{prefix}', candidate))

    best_name = None
    best_filtered = None
    best_matched = -1
    for name, cand in variants:
        filtered = {k: v for k, v in cand.items() if k in current and current[k].shape == v.shape}
        if len(filtered) > best_matched:
            best_name = name
            best_filtered = filtered
            best_matched = len(filtered)

    missing, unexpected = module.load_state_dict(best_filtered, strict=False)
    summary = {
        'variant': best_name,
        'matched_keys': len(best_filtered),
        'missing_keys': len(missing),
        'unexpected_keys': len(unexpected),
    }
    return summary, missing, unexpected


class _ConvNeXtV2MultiScaleBackbone(nn.Module):
    def __init__(self, model_name: str, fallback_model_name: str, fpn_dim: int = 128, groups: int = 8):
        super().__init__()
        self.model_name = None
        self.backbone = None
        errors = []
        for candidate_name in [model_name, fallback_model_name]:
            try:
                self.backbone = timm.create_model(candidate_name, pretrained=False)
                self.model_name = candidate_name
                break
            except Exception as e:
                errors.append((candidate_name, repr(e)))
        if self.backbone is None:
            raise RuntimeError(f'Failed to create ConvNeXtV2 backbone: {errors}')

        stages = getattr(self.backbone, 'stages', None)
        stem = getattr(self.backbone, 'stem', None)
        if stages is None or stem is None:
            raise RuntimeError('ConvNeXt backbone is expected to expose .stem and .stages')
        if len(stages) < 4:
            raise RuntimeError(f'Expected 4 ConvNeXt stages, got {len(stages)}')

        channels = [256, 512, 1024]
        self.laterals = nn.ModuleList([nn.Conv2d(c, fpn_dim, 1) for c in channels])
        self.smooth16 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.smooth8 = ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True)
        self.out_proj = nn.Sequential(
            ConvGNAct(fpn_dim * 3, fpn_dim, k=1, s=1, p=0, groups=groups, act=True),
            ConvGNAct(fpn_dim, fpn_dim, k=3, s=1, p=1, groups=groups, act=True),
        )

    def freeze_all_stages(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_stages(self, n_last_stages=2):
        self.freeze_all_stages()
        stages = getattr(self.backbone, 'stages', None)
        if stages is None:
            raise RuntimeError('ConvNeXt backbone has no .stages attribute for staged unfreeze')
        for stage in list(stages)[-n_last_stages:]:
            for p in stage.parameters():
                p.requires_grad = True
        for aux_name in ['norm_pre', 'head', 'norm']:
            aux = getattr(self.backbone, aux_name, None)
            if aux is not None:
                for p in aux.parameters():
                    p.requires_grad = True

    def forward(self, x):
        x = self.backbone.stem(x)
        x = self.backbone.stages[0](x)
        feat_s8 = self.backbone.stages[1](x)
        feat_s16 = self.backbone.stages[2](feat_s8)
        feat_s32 = self.backbone.stages[3](feat_s16)
        lat8 = self.laterals[0](feat_s8)
        lat16 = self.laterals[1](feat_s16)
        p32 = self.laterals[2](feat_s32)
        p16 = self.smooth16(lat16 + F.interpolate(p32, size=lat16.shape[-2:], mode='bilinear', align_corners=False))
        p8 = self.smooth8(lat8 + F.interpolate(p16, size=lat8.shape[-2:], mode='bilinear', align_corners=False))
        p16_up = F.interpolate(p16, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        p32_up = F.interpolate(p32, size=p8.shape[-2:], mode='bilinear', align_corners=False)
        fused = self.out_proj(torch.cat([p8, p16_up, p32_up], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'fused': fused,
        }


class MultiCamBEVv8ConvNeXtV2LSSClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 model_name: str = 'convnextv2_base.fcmae_ft_in22k_in1k_384',
                 fallback_model_name: str = 'convnextv2_base.fcmae_ft_in22k_in1k',
                 fpn_dim: int = 128,
                 context_dim: int = 80,
                 depth_bins: int = 24,
                 depth_min: float = 1.0,
                 depth_max: float = 80.0,
                 world_z_min: float = -2.0,
                 world_z_max: float = 4.5,
                 bev_base_channels: int = 96,
                 bev_gn_groups: int = 8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.backbone = _ConvNeXtV2MultiScaleBackbone(
            model_name=model_name,
            fallback_model_name=fallback_model_name,
            fpn_dim=fpn_dim,
            groups=bev_gn_groups,
        )
        if freeze_backbone:
            self.backbone.freeze_all_stages()

        self.view_transform = LSSViewTransform2D(
            in_c=fpn_dim,
            context_c=context_dim,
            depth_bins=depth_bins,
            depth_min=depth_min,
            depth_max=depth_max,
            bev_h=BEV_H,
            bev_w=BEV_W,
            bev_res=BEV_RES,
            x_range=X_RANGE,
            y_range=Y_RANGE,
            z_min=world_z_min,
            z_max=world_z_max,
            groups=bev_gn_groups,
        )
        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )
        self.bev_decoder = StrongBEVEncoderDecoder(
            in_c=context_dim + rover_cond_dim,
            base_c=bev_base_channels,
            groups=bev_gn_groups,
        )

    def forward_debug(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        assert N == self.n_cameras
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        feat_s8 = back['feat_s8'].reshape(B, N, back['feat_s8'].shape[1], back['feat_s8'].shape[2], back['feat_s8'].shape[3])
        feat_s16 = back['feat_s16'].reshape(B, N, back['feat_s16'].shape[1], back['feat_s16'].shape[2], back['feat_s16'].shape[3])
        feat_s32 = back['feat_s32'].reshape(B, N, back['feat_s32'].shape[1], back['feat_s32'].shape[2], back['feat_s32'].shape[3])
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev, vt_debug = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return {
            'feat_s8': feat_s8,
            'feat_s16': feat_s16,
            'feat_s32': feat_s32,
            'image_fused': fused,
            'depth_prob': vt_debug['depth_prob'],
            'bev_raw': vt_debug['bev'],
            'valid_ratio': vt_debug['valid_ratio'],
            'logits': logits,
        }

    def forward(self, images, intrinsics, car2cams, rover_ids):
        dbg = self.forward_debug(images, intrinsics, car2cams, rover_ids)
        return torch.nan_to_num(dbg['logits'], nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
convnext_ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
convnext_ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
convnext_ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

convnext_train_sampler_warm = None
convnext_train_sampler_aug = None
if is_dist_enabled():
    convnext_train_sampler_warm = DistributedSampler(convnext_ds_train_warm, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)
    convnext_train_sampler_aug = DistributedSampler(convnext_ds_train_aug, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)

convnext_loader_warm = DataLoader(
    convnext_ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=(convnext_train_sampler_warm is None),
    sampler=convnext_train_sampler_warm,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
convnext_loader_train = DataLoader(
    convnext_ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=(convnext_train_sampler_aug is None),
    sampler=convnext_train_sampler_aug,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
convnext_loader_val = None
if is_main_process():
    convnext_loader_val = DataLoader(
        convnext_ds_val,
        batch_size=cfg['val_batch_size'],
        shuffle=False,
        num_workers=cfg['val_num_workers'],
        pin_memory=(device.type == 'cuda'),
    )
    print('convnext loader_train batch_size:', convnext_loader_train.batch_size)
    print('convnext loader_val batch_size:', convnext_loader_val.batch_size)

convnext_base_model = MultiCamBEVv8ConvNeXtV2LSSClean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    model_name=cfg['convnext_model_name'],
    fallback_model_name=cfg['convnext_fallback_model_name'],
    fpn_dim=cfg['fpn_dim'],
    context_dim=cfg['context_dim'],
    depth_bins=cfg['depth_bins'],
    depth_min=cfg['depth_min'],
    depth_max=cfg['depth_max'],
    world_z_min=cfg['world_z_min'],
    world_z_max=cfg['world_z_max'],
    bev_base_channels=cfg['bev_base_channels'],
    bev_gn_groups=cfg['bev_gn_groups'],
).to(device)

def convnext_freeze_all_backbone(model):
    unwrap_model(model).backbone.freeze_all_stages()

def convnext_unfreeze_last_stages(model, n_last_stages=2):
    unwrap_model(model).backbone.unfreeze_last_stages(n_last_stages)

if is_dist_enabled():
    convnext_model = DDP(
        convnext_base_model,
        device_ids=[get_local_rank()],
        output_device=get_local_rank(),
        broadcast_buffers=False,
        find_unused_parameters=False,
    )
else:
    convnext_model = convnext_base_model

convnext_core_model = unwrap_model(convnext_model)
convnext_backbone_params = []
convnext_embed_params = []
convnext_image_neck_params = []
convnext_other_params = []
for name, p in convnext_core_model.named_parameters():
    if name.startswith('backbone.backbone.'):
        convnext_backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        convnext_embed_params.append(p)
    elif name.startswith('backbone.laterals.') or name.startswith('backbone.smooth') or name.startswith('backbone.out_proj.'):
        convnext_image_neck_params.append(p)
    else:
        convnext_other_params.append(p)

convnext_optimizer = torch.optim.AdamW([
    {'params': convnext_backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': convnext_image_neck_params, 'lr': cfg['lr_image_neck'], 'weight_decay': cfg['weight_decay']},
    {'params': convnext_other_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['weight_decay']},
    {'params': convnext_embed_params, 'lr': cfg['lr_lss_bev'], 'weight_decay': cfg['embed_weight_decay']},
])
convnext_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(convnext_optimizer, T_max=max(cfg['epochs'], 1), eta_min=1e-6)
convnext_loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
convnext_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda' and cfg['use_amp']))

convnext_ema_model = copy.deepcopy(convnext_core_model).to(device).eval()
for p in convnext_ema_model.parameters():
    p.requires_grad = False

@torch.no_grad()
def update_ema(ema_model, model, decay):
    src = unwrap_model(model)
    ema_params = dict(ema_model.named_parameters())
    src_params = dict(src.named_parameters())
    for name, param in src_params.items():
        ema_params[name].mul_(decay).add_(param.data, alpha=1.0 - decay)
    ema_buffers = dict(ema_model.named_buffers())
    src_buffers = dict(src.named_buffers())
    for name, buf in src_buffers.items():
        ema_buffers[name].copy_(buf)

resume_ckpt = torch.load(local_last, map_location='cpu')
convnext_core_model.load_state_dict(resume_ckpt['model'], strict=True)
if 'ema' in resume_ckpt:
    convnext_ema_model.load_state_dict(resume_ckpt['ema'], strict=True)
if 'optimizer' in resume_ckpt:
    convnext_optimizer.load_state_dict(resume_ckpt['optimizer'])
if 'scaler' in resume_ckpt and resume_ckpt['scaler'] is not None:
    convnext_scaler.load_state_dict(resume_ckpt['scaler'])

# Tail continuation: replace scheduler with a short 10-epoch cosine tail.
convnext_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(convnext_optimizer, T_max=cfg['epochs'], eta_min=1e-6)

convnext_start_epoch = int(resume_ckpt.get('epoch', -1)) + 1
convnext_total_epochs = convnext_start_epoch + int(cfg['epochs'])
convnext_best_iou = float(resume_ckpt.get('best_iou', -1.0))
convnext_best_ema_iou = float(resume_ckpt.get('best_ema_iou', -1.0))
convnext_log = []
log_csv_path = local_last.parent / 'log.csv'
if log_csv_path.exists():
    convnext_log = pd.read_csv(log_csv_path).to_dict('records')
else:
    run_log = RUN_DIR / 'log.csv'
    if run_log.exists():
        convnext_log = pd.read_csv(run_log).to_dict('records')

convnext_start = time.time() - (float(convnext_log[-1].get('minutes', 0.0)) * 60.0 if len(convnext_log) else 0.0)
convnext_unfreeze_last_stages(convnext_model, n_last_stages=cfg['unfreeze_last_stages'])

if is_main_process():
    print('resume start epoch:', convnext_start_epoch)
    print('resume through epoch:', convnext_total_epochs - 1)
    print('best_iou:', convnext_best_iou, '| best_ema_iou:', convnext_best_ema_iou)
    print('params M total:', sum(p.numel() for p in convnext_core_model.parameters()) / 1e6)
    print('params M trainable:', sum(p.numel() for p in convnext_core_model.parameters() if p.requires_grad) / 1e6)

    with torch.no_grad():
        batch = next(iter(convnext_loader_train))
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        dbg = convnext_core_model.forward_debug(images, intr, c2c, rover_id)
        print('sanity logits:', tuple(dbg['logits'].shape), 'finite:', torch.isfinite(dbg['logits']).all().item())
        print('sanity valid_ratio:', dbg['valid_ratio'])

cleanup_cuda()
barrier()


In [ ]:
WANDB_RUN_ID_PATH = RUN_DIR / 'wandb_run_id.txt'
if WANDB_RUN_ID_PATH.exists():
    wandb_run_id = WANDB_RUN_ID_PATH.read_text().strip()
else:
    wandb_run_id = uuid.uuid4().hex
    WANDB_RUN_ID_PATH.write_text(wandb_run_id)

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    id=wandb_run_id,
    resume='allow',
    name=f'v8_convnext_wandb_resume_from_e{convnext_start_epoch - 1}',
    dir=str(RUN_DIR),
    config=dict(cfg),
)

wandb.define_metric('epoch')
wandb.define_metric('global_step')
wandb.define_metric('train/*', step_metric='global_step')
wandb.define_metric('val/*', step_metric='epoch')
wandb.define_metric('lr/*', step_metric='epoch')

def upload_checkpoint_bundle(epoch: int):
    artifact = wandb.Artifact(
        name='v8-convnext-checkpoints',
        type='model',
        metadata={
            'epoch': int(epoch),
            'best_iou': float(convnext_best_iou),
            'best_ema_iou': float(convnext_best_ema_iou),
        },
    )
    for fname in ['best.pt', 'ema_best.pt', 'last.pt', 'log.csv', 'config.json']:
        path = RUN_DIR / fname
        if path.exists():
            artifact.add_file(str(path), name=fname)
    wandb.log_artifact(artifact, aliases=['latest', f'epoch-{epoch:03d}'])

# Seed local files with downloaded artifacts.
for src, dst in [
    (local_best, RUN_DIR / 'best.pt'),
    (local_ema, RUN_DIR / 'ema_best.pt'),
    (local_last, RUN_DIR / 'last.pt'),
]:
    dst.write_bytes(src.read_bytes())
if log_csv_path.exists():
    (RUN_DIR / 'log.csv').write_bytes(log_csv_path.read_bytes())

if len(convnext_log):
    for row in convnext_log:
        wandb.log({
            'epoch': int(row['epoch']),
            'train/loss_epoch': float(row['loss']) if row['loss'] is not None else None,
            'val/iou_05': float(row['val_iou_05']) if row['val_iou_05'] is not None else None,
            'val/iou_05_ema': float(row['val_iou_05_ema']) if row['val_iou_05_ema'] is not None else None,
        })

global_step = convnext_start_epoch * max(len(convnext_loader_train), 1)

for epoch in range(convnext_start_epoch, convnext_total_epochs):
    if convnext_train_sampler_warm is not None:
        convnext_train_sampler_warm.set_epoch(epoch)
    if convnext_train_sampler_aug is not None:
        convnext_train_sampler_aug.set_epoch(epoch)

    convnext_model.train()
    losses = []
    convnext_optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(convnext_loader_train, desc=f'convnext ep{epoch:02d} [AUG]', disable=not is_main_process())

    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = convnext_model(images, intr, c2c, rover_id)
        logits = logits.float()
        loss, parts = convnext_loss_fn(logits, gt)
        loss = loss / cfg['grad_accum']

        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite ConvNeXt loss at epoch={epoch} step={step}: {loss.item()}')

        convnext_scaler.scale(loss).backward()

        if (step + 1) % cfg['grad_accum'] == 0:
            convnext_scaler.unscale_(convnext_optimizer)
            torch.nn.utils.clip_grad_norm_(convnext_core_model.parameters(), max_norm=1.0)
            convnext_scaler.step(convnext_optimizer)
            convnext_scaler.update()
            convnext_optimizer.zero_grad(set_to_none=True)
            update_ema(convnext_ema_model, convnext_model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        global_step += 1

        if is_main_process() and step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")
            wandb.log({
                'global_step': global_step,
                'train/loss_step': float(np.mean(losses[-50:])),
                'train/bce_step': float(parts['bce']),
            })

    if len(convnext_loader_train) % cfg['grad_accum'] != 0:
        convnext_scaler.unscale_(convnext_optimizer)
        torch.nn.utils.clip_grad_norm_(convnext_core_model.parameters(), max_norm=1.0)
        convnext_scaler.step(convnext_optimizer)
        convnext_scaler.update()
        convnext_optimizer.zero_grad(set_to_none=True)
        update_ema(convnext_ema_model, convnext_model, cfg['ema_decay'])

    convnext_scheduler.step()
    barrier()

    if is_main_process():
        cleanup_cuda()
        print('start convnext val model @0.5')
        val_iou_05 = evaluate_iou(convnext_core_model, convnext_loader_val, threshold=0.5, desc=f'convnext val model ep{epoch:02d}')

        cleanup_cuda()
        print('start convnext val ema @0.5')
        val_iou_05_ema = evaluate_iou(convnext_ema_model, convnext_loader_val, threshold=0.5, desc=f'convnext val ema ep{epoch:02d}')
        cleanup_cuda()

        elapsed = (time.time() - convnext_start) / 60
        backbone_grad_enabled = any(p.requires_grad for p in convnext_core_model.backbone.backbone.parameters())

        row = {
            'epoch': epoch,
            'loss': float(np.mean(losses)) if len(losses) else None,
            'val_iou_05': float(val_iou_05),
            'val_iou_05_ema': float(val_iou_05_ema),
            'backbone_trainable': bool(backbone_grad_enabled),
            'minutes': float(elapsed),
        }

        print(
            f"convnext ep{epoch:02d} | loss {np.mean(losses):.3f} | "
            f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
            f"backbone_trainable={backbone_grad_enabled} | {elapsed:.1f}m"
        )

        if val_iou_05 > convnext_best_iou:
            convnext_best_iou = val_iou_05
            torch.save({
                'model_type': 'v8_convnextv2_fcmae_lss_bev_cleaned',
                'model': convnext_core_model.state_dict(),
                'epoch': epoch,
                'best_iou': float(val_iou_05),
                'best_t': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new convnext best model:', val_iou_05)

        if val_iou_05_ema > convnext_best_ema_iou:
            convnext_best_ema_iou = val_iou_05_ema
            torch.save({
                'model_type': 'v8_convnextv2_fcmae_lss_bev_cleaned',
                'model': convnext_core_model.state_dict(),
                'ema': convnext_ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': float(val_iou_05_ema),
                'best_t_ema': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new convnext best ema:', val_iou_05_ema)

        convnext_log.append(row)
        pd.DataFrame(convnext_log).to_csv(RUN_DIR / 'log.csv', index=False)

        torch.save({
            'model_type': 'v8_convnextv2_fcmae_lss_bev_cleaned',
            'model': convnext_core_model.state_dict(),
            'ema': convnext_ema_model.state_dict(),
            'optimizer': convnext_optimizer.state_dict(),
            'scheduler': convnext_scheduler.state_dict(),
            'scaler': convnext_scaler.state_dict() if convnext_scaler is not None else None,
            'epoch': epoch,
            'best_iou': float(convnext_best_iou),
            'best_ema_iou': float(convnext_best_ema_iou),
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'last.pt')

        wandb.log({
            'epoch': epoch,
            'train/loss_epoch': float(np.mean(losses)),
            'val/iou_05': float(val_iou_05),
            'val/iou_05_ema': float(val_iou_05_ema),
            'lr/backbone': float(convnext_optimizer.param_groups[0]['lr']),
            'lr/image_neck': float(convnext_optimizer.param_groups[1]['lr']),
            'lr/lss_bev': float(convnext_optimizer.param_groups[2]['lr']),
            'system/minutes': float(elapsed),
        })

        upload_checkpoint_bundle(epoch)

    barrier()

if is_main_process():
    wandb.finish()


### Notes

- Для resume достаточно `last.pt`, но notebook также скачивает `best.pt` и `ema_best.pt` из `wandb`, чтобы локально восстановить весь набор чекпоинтов.
- Для dataset ожидаются папки `autonomy_yandex_dataset_*_kaggle_safe` в `/content`. При другом layout достаточно поправить `DATA_TRAIN`, `DATA_VAL`, `DATA_TEST` в первой ячейке.
- Каждую эпоху в `wandb` загружается bundle artifact `v8-convnext-checkpoints` с `best.pt`, `ema_best.pt`, `last.pt`, `log.csv`, `config.json`.
